# DIAGNÓSTICO REUTERS — VERSIÓN CANÓNICA

**Versión:** `v0.10-reuters-retrieval-debug`  
**Última modificación:** `2026-09-11 20:21 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo LLM:** `gpt-4.1-mini`

## Qué sabemos ya por la ejecución de v0.9

- Hosted `web_search` **sí funciona en general**: A1 recuperó 12 fuentes y A2 recuperó 15.
- Las pruebas Reuters A3, A4 y A5 devolvieron `sources=None` y 0 URLs.
- A6 falló porque `gpt-4.1-mini` no admite `filters` en `web_search`. Ese error era del test, no del agente.
- Por tanto, ahora comprobamos si Reuters falla solo en *discovery/indexación* o también al acceder a una URL Reuters exacta.


## 1. Instalar dependencias

In [ ]:
!pip install -U openai openai-agents ddgs requests -q


## 2. Entorno y helpers

In [ ]:
import os, sys, importlib.metadata as im, requests
from urllib.parse import urlsplit
from google.colab import userdata
from openai import OpenAI
from ddgs import DDGS

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
client = OpenAI()

print('Python:', sys.version)
print('openai:', im.version('openai'))
print('openai-agents:', im.version('openai-agents'))
print('ddgs:', im.version('ddgs'))
print('API key presente:', bool(os.environ.get('OPENAI_API_KEY')))

REUTERS_URL = 'https://www.reuters.com/business/openai-launches-chatgpt-financial-services-industry-2026-09-10/'

def to_dict(obj):
    return obj.model_dump() if hasattr(obj, 'model_dump') else obj

def collect_urls(obj):
    obj = to_dict(obj)
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in {'url', 'source_url', 'source_website_url'} and isinstance(v, str) and v.startswith('http'):
                found.append(v)
            found.extend(collect_urls(v))
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            found.extend(collect_urls(v))
    return list(dict.fromkeys(found))

def run_openai_probe(label, query):
    print('\n' + '#' * 90)
    print(label)
    print('QUERY:', query)
    resp = client.responses.create(
        model='gpt-4.1-mini',
        tools=[{'type':'web_search','search_context_size':'high','external_web_access':True}],
        tool_choice='required',
        include=['web_search_call.action.sources'],
        input=query,
    )
    print('OUTPUT TEXT:', resp.output_text)
    calls=[x for x in resp.output if getattr(x,'type',None)=='web_search_call']
    for i,c in enumerate(calls,1):
        a=getattr(c,'action',None)
        print(f'CALL {i} status:', getattr(c,'status',None))
        print('  query:', getattr(a,'query',None))
        print('  sources:', getattr(a,'sources',None))
    urls=collect_urls(resp)
    reuters=[u for u in urls if 'reuters.com' in (urlsplit(u).hostname or '').lower()]
    print('TOTAL URLS:', len(urls))
    for u in urls[:30]: print('  URL:',u)
    print('REUTERS URLS:', len(reuters))
    for u in reuters: print('  REUTERS:',u)
    return {'response':resp,'urls':urls,'reuters':reuters}


## 3. B1 — Control general: web_search debe devolver fuentes

In [ ]:
b1 = run_openai_probe(
    'B1 — CONTROL GENERAL',
    'Find the official OpenAI page about new tools for building agents. Cite the source.'
)


## 4. B2 — Reuters por búsqueda `site:reuters.com`

No usamos `filters`, porque la ejecución anterior demostró que `gpt-4.1-mini` los rechaza.

In [ ]:
b2 = run_openai_probe(
    'B2 — SITE REUTERS',
    "Search the live web for: site:reuters.com 'OpenAI launches ChatGPT for financial services industry' '2026-09-10'. Return the direct Reuters URL and cite it."
)


## 5. B3 — Dar al hosted web_search la URL Reuters exacta

Esta prueba distingue entre **no indexa Reuters** y **no puede acceder a Reuters ni con URL conocida**.

In [ ]:
b3 = run_openai_probe(
    'B3 — URL REUTERS EXACTA',
    f'Open or verify this exact Reuters URL and tell me the article headline and publication date. Cite the same URL: {REUTERS_URL}'
)


## 6. B4 — Acceso HTTP directo desde Colab a Reuters

In [ ]:
print('URL:', REUTERS_URL)
try:
    r = requests.get(
        REUTERS_URL,
        headers={'User-Agent':'Mozilla/5.0'},
        timeout=20,
        allow_redirects=True,
    )
    print('HTTP status:', r.status_code)
    print('Final URL:', r.url)
    print('Content-Type:', r.headers.get('content-type'))
    print('Bytes:', len(r.content))
    print('Primeros 300 caracteres:', r.text[:300].replace('\n',' '))
except Exception as e:
    print('HTTP ERROR:', type(e).__name__, str(e)[:1000])


## 7. B5 — Descubrimiento alternativo con DuckDuckGo

Es solo un smoke test. Si devuelve Reuters, podemos convertirlo después en un `function_tool` del agente y mantener `gpt-4.1-mini` para síntesis y evaluación.

In [ ]:
ddg_results=[]
try:
    ddg_results = list(DDGS().text(
        "site:reuters.com OpenAI ChatGPT financial services industry September 10 2026",
        max_results=10,
    ))
    print('DDG results:', len(ddg_results))
    for x in ddg_results:
        print('TITLE:', x.get('title'))
        print('URL:  ', x.get('href'))
        print()
except Exception as e:
    print('DDG ERROR:', type(e).__name__, str(e)[:1000])

ddg_reuters=[x.get('href') for x in ddg_results if x.get('href') and 'reuters.com' in (urlsplit(x.get('href')).hostname or '').lower()]
print('DDG Reuters URLs:', len(ddg_reuters))
for u in ddg_reuters: print('  REUTERS:',u)


## 8. Diagnóstico automático

In [ ]:
print('\n'+'='*90)
print('DIAGNÓSTICO FINAL v0.10')
print('B1 URLs generales:', len(b1['urls']))
print('B2 Reuters vía site:', len(b2['reuters']))
print('B3 Reuters con URL exacta:', len(b3['reuters']))
print('B5 Reuters vía DDG:', len(ddg_reuters))

if len(b1['urls']) == 0:
    print('RESULTADO: hosted web_search no está funcionando en general.')
elif len(b2['reuters']) > 0:
    print('RESULTADO: Reuters sí es descubrible por hosted web_search. Podemos volver al agente.')
elif len(b3['reuters']) > 0:
    print('RESULTADO: hosted web_search puede acceder a Reuters si conoce la URL, pero no la descubre bien.')
    print('SOLUCIÓN: usar otra capa para descubrir URLs y OpenAI para leer/sintetizar/evaluar.')
elif len(ddg_reuters) > 0:
    print('RESULTADO: OpenAI hosted web_search no recupera Reuters, pero DDG sí lo descubre.')
    print('SOLUCIÓN PROPUESTA: implementar búsqueda Reuters como function_tool con DDG y mantener gpt-4.1-mini.')
else:
    print('RESULTADO: Reuters no aparece ni en hosted web_search ni en DDG en estas pruebas.')
    print('Habrá que usar otro proveedor de búsqueda/news para Reuters.')
